
# 2. Construção da camada Gold

In [0]:
%sql

-- Verifica as tabelas atualmente disponíveis na camada Gold.

SHOW TABLES IN mvp.gold;

database,tableName,isTemporary
gold,dim_orgao,false
gold,dim_regiao,false
gold,dim_tempo,false
gold,fato_bolsas,false
gold,fato_orcamento,false


In [0]:
%sql

-- Dimensão temporal utilizada pelas duas tabelas fato.
CREATE OR REPLACE TABLE mvp.gold.dim_tempo
USING DELTA
AS
SELECT
    CAST(ROW_NUMBER() OVER (ORDER BY ano) AS INT) AS id_tempo,
    ano,
    CASE
        WHEN ano BETWEEN 2013 AND 2016 THEN '2013-2016'
        WHEN ano BETWEEN 2017 AND 2020 THEN '2017-2020'
    END AS periodo
FROM (
    SELECT DISTINCT AN_REFERENCIA AS ano
    FROM mvp.silver.bolsas_capes
    WHERE AN_REFERENCIA BETWEEN 2013 AND 2020
);


-- Identifica os dois órgãos utilizados na análise orçamentária.
CREATE OR REPLACE TABLE mvp.gold.dim_orgao
USING DELTA
AS
SELECT * FROM VALUES
    (1, '26291', 'CAPES', 'Fundação'),
    (2, '26000', 'MEC', 'Ministério')
AS dim_orgao (
    id_orgao,
    codigo_orgao,
    nome_orgao,
    tipo_orgao
);


-- Regiões utilizadas na análise da distribuição das bolsas.
CREATE OR REPLACE TABLE mvp.gold.dim_regiao
USING DELTA
AS
SELECT
    CAST(ROW_NUMBER() OVER (ORDER BY nome_regiao) AS INT) AS id_regiao,
    nome_regiao
FROM (
    SELECT DISTINCT
        NM_REGIAO_IES_CAPES AS nome_regiao
    FROM mvp.silver.bolsas_capes
    WHERE NM_REGIAO_IES_CAPES IS NOT NULL
);

num_affected_rows,num_inserted_rows


In [0]:
%sql

-- Cria a fato de orçamento por ano e órgão.
CREATE OR REPLACE TABLE mvp.gold.fato_orcamento
USING DELTA
AS

-- Orçamento do MEC
SELECT
    t.id_tempo,
    o.id_orgao,
    m.DOTACAO_INICIAL AS dotacao_inicial,
    m.DOTACAO_ATUAL AS dotacao_atual,
    m.EMPENHADO AS empenhado,
    m.LIQUIDADO AS liquidado,
    m.PAGO AS pago

FROM mvp.silver.orcamento_mec m

INNER JOIN mvp.gold.dim_tempo t
    ON m.ANO = t.ano

INNER JOIN mvp.gold.dim_orgao o
    ON o.nome_orgao = 'MEC'

WHERE m.ANO BETWEEN 2013 AND 2020


UNION ALL


-- Orçamento da CAPES
SELECT
    t.id_tempo,
    o.id_orgao,
    c.DOTACAO_INICIAL,
    c.DOTACAO_ATUAL,
    c.EMPENHADO,
    c.LIQUIDADO,
    c.PAGO

FROM mvp.silver.orcamento_mec_unidades c

INNER JOIN mvp.gold.dim_tempo t
    ON c.ANO = t.ano

INNER JOIN mvp.gold.dim_orgao o
    ON o.nome_orgao = 'CAPES'

WHERE c.ANO BETWEEN 2013 AND 2020
  AND c.UNIDADE_ORCAMENTARIA LIKE '26291%';


-- Cria a fato de bolsas agregada por ano e região.
CREATE OR REPLACE TABLE mvp.gold.fato_bolsas
USING DELTA
AS

SELECT
    t.id_tempo,
    r.id_regiao,
    SUM(b.QT_BOLSAS_CONCEDIDAS) AS quantidade_bolsas

FROM mvp.silver.bolsas_capes b

INNER JOIN mvp.gold.dim_tempo t
    ON b.AN_REFERENCIA = t.ano

INNER JOIN mvp.gold.dim_regiao r
    ON b.NM_REGIAO_IES_CAPES = r.nome_regiao

WHERE b.AN_REFERENCIA BETWEEN 2013 AND 2020

GROUP BY
    t.id_tempo,
    r.id_regiao;

num_affected_rows,num_inserted_rows


## 2.1 Validação da camada Gold

Após a construção das tabelas fato e dimensão, foram realizadas verificações para confirmar a preservação dos dados utilizados nas análises e identificar possíveis perdas durante os relacionamentos e agregações.

In [0]:
%sql

-- Valida a quantidade de registros e o total de bolsas na Gold.
SELECT
    'Registros fato_orcamento' AS validacao,
    COUNT(*) AS valor
FROM mvp.gold.fato_orcamento

UNION ALL

SELECT
    'Registros fato_bolsas',
    COUNT(*)
FROM mvp.gold.fato_bolsas

UNION ALL

SELECT
    'Total bolsas Silver',
    SUM(QT_BOLSAS_CONCEDIDAS)
FROM mvp.silver.bolsas_capes

UNION ALL

SELECT
    'Total bolsas Gold',
    SUM(quantidade_bolsas)
FROM mvp.gold.fato_bolsas;

validacao,valor
Total bolsas Silver,789773
Total bolsas Gold,789773
Registros fato_orcamento,16
Registros fato_bolsas,40


## Questão 1 — Como evoluiu o orçamento da CAPES entre 2013 e 2020, especialmente antes e depois de 2016?

In [0]:
%sql

-- Analisa a evolução anual do orçamento da CAPES entre 2013 e 2020.

SELECT
    t.ano,
    ROUND(f.dotacao_inicial / 1000000000, 2) AS dotacao_inicial_bi,
    ROUND(f.dotacao_atual / 1000000000, 2) AS dotacao_atual_bi,
    ROUND(f.empenhado / 1000000000, 2) AS empenhado_bi,
    ROUND(f.liquidado / 1000000000, 2) AS liquidado_bi,
    ROUND(f.pago / 1000000000, 2) AS pago_bi

FROM mvp.gold.fato_orcamento f

INNER JOIN mvp.gold.dim_tempo t
    ON f.id_tempo = t.id_tempo

INNER JOIN mvp.gold.dim_orgao o
    ON f.id_orgao = o.id_orgao

WHERE o.nome_orgao = 'CAPES'

ORDER BY t.ano;

ano,dotacao_inicial_bi,dotacao_atual_bi,empenhado_bi,liquidado_bi,pago_bi
2013,4.24,5.30,4.83,4.28,4.26
2014,5.08,6.07,5.58,5.42,5.40
2015,6.27,7.43,7.77,7.66,7.65
2016,5.29,5.90,5.71,5.06,5.06
2017,4.96,4.96,4.71,3.94,3.94
2018,3.98,3.84,3.86,3.35,3.33
2019,4.25,4.19,4.20,3.82,3.82
2020,2.84,3.55,3.53,3.04,3.03


In [0]:
%sql
-- Compara as médias orçamentárias da CAPES entre 2013-2016 e 2017-2020.
SELECT
    t.periodo,
    ROUND(AVG(f.dotacao_inicial) / 1000000000, 2)
        AS media_dotacao_inicial_bi,

    ROUND(AVG(f.dotacao_atual) / 1000000000, 2)
        AS media_dotacao_atual_bi,

    ROUND(AVG(f.empenhado) / 1000000000, 2)
        AS media_empenhado_bi,

    ROUND(AVG(f.liquidado) / 1000000000, 2)
        AS media_liquidado_bi,

    ROUND(AVG(f.pago) / 1000000000, 2)
        AS media_pago_bi

FROM mvp.gold.fato_orcamento f

INNER JOIN mvp.gold.dim_tempo t
    ON f.id_tempo = t.id_tempo

INNER JOIN mvp.gold.dim_orgao o
    ON f.id_orgao = o.id_orgao

WHERE o.nome_orgao = 'CAPES'
GROUP BY t.periodo
ORDER BY t.periodo;

periodo,media_dotacao_inicial_bi,media_dotacao_atual_bi,media_empenhado_bi,media_liquidado_bi,media_pago_bi
2013-2016,5.22,6.18,5.97,5.61,5.59
2017-2020,4.01,4.14,4.07,3.54,3.53


## Questão 2 — As alterações observadas no orçamento da CAPES acompanharam o comportamento do orçamento do MEC ou a CAPES apresentou uma dinâmica distinta entre 2013 e 2020?

In [0]:
%sql

-- Compara a evolução anual da dotação inicial da CAPES e do MEC.

WITH orcamento AS (
    SELECT
        t.ano,
        o.nome_orgao,
        f.dotacao_inicial
    FROM mvp.gold.fato_orcamento f

    INNER JOIN mvp.gold.dim_tempo t
        ON f.id_tempo = t.id_tempo

    INNER JOIN mvp.gold.dim_orgao o
        ON f.id_orgao = o.id_orgao
),

comparacao AS (
    SELECT
        ano,

        MAX(CASE
            WHEN nome_orgao = 'CAPES'
            THEN dotacao_inicial
        END) AS capes_dotacao,

        MAX(CASE
            WHEN nome_orgao = 'MEC'
            THEN dotacao_inicial
        END) AS mec_dotacao

    FROM orcamento
    GROUP BY ano
)

SELECT
    ano,

    ROUND(capes_dotacao / 1000000000, 2)
        AS capes_dotacao_bi,

    ROUND(
        (capes_dotacao /
        LAG(capes_dotacao) OVER (ORDER BY ano) - 1) * 100,
        2
    ) AS capes_variacao_pct,

    ROUND(mec_dotacao / 1000000000, 2)
        AS mec_dotacao_bi,

    ROUND(
        (mec_dotacao /
        LAG(mec_dotacao) OVER (ORDER BY ano) - 1) * 100,
        2
    ) AS mec_variacao_pct

FROM comparacao

ORDER BY ano;

ano,capes_dotacao_bi,capes_variacao_pct,mec_dotacao_bi,mec_variacao_pct
2013,4.24,null,81.29,null
2014,5.08,19.67,94.49,16.24
2015,6.27,23.50,103.36,9.39
2016,5.29,-15.59,99.79,-3.46
2017,4.96,-6.27,107.52,7.74
2018,3.98,-19.85,107.55,0.03
2019,4.25,6.96,122.95,14.33
2020,2.84,-33.12,103.11,-16.13


In [0]:
%sql

-- Compara a variação da dotação inicial da CAPES e do MEC entre 2016 e 2020.

WITH valores AS (
    SELECT
        t.ano,
        o.nome_orgao,
        f.dotacao_inicial
    FROM mvp.gold.fato_orcamento f

    INNER JOIN mvp.gold.dim_tempo t
        ON f.id_tempo = t.id_tempo

    INNER JOIN mvp.gold.dim_orgao o
        ON f.id_orgao = o.id_orgao

    WHERE t.ano IN (2016, 2020)
),

comparacao AS (
    SELECT
        nome_orgao,

        MAX(CASE WHEN ano = 2016
            THEN dotacao_inicial END) AS valor_2016,

        MAX(CASE WHEN ano = 2020
            THEN dotacao_inicial END) AS valor_2020

    FROM valores

    GROUP BY nome_orgao
)

SELECT
    nome_orgao,

    ROUND(valor_2016 / 1000000000, 2) AS dotacao_2016_bi,
    ROUND(valor_2020 / 1000000000, 2) AS dotacao_2020_bi,

    ROUND(
        (valor_2020 / valor_2016 - 1) * 100,
        2
    ) AS variacao_2016_2020_pct

FROM comparacao

ORDER BY nome_orgao;

nome_orgao,dotacao_2016_bi,dotacao_2020_bi,variacao_2016_2020_pct
CAPES,5.29,2.84,-46.26
MEC,99.79,103.11,3.33


## Questão 3 — Como evoluiu a participação do orçamento da CAPES em relação ao orçamento do MEC entre 2013 e 2020, especialmente na comparação entre os períodos anterior e posterior a 2016?

In [0]:
%sql

-- Calcula a participação da dotação inicial da CAPES em relação à dotação inicial total do MEC.
WITH orcamento AS (
    SELECT
        t.ano,
        o.nome_orgao,
        f.dotacao_inicial
    FROM mvp.gold.fato_orcamento f

    INNER JOIN mvp.gold.dim_tempo t
        ON f.id_tempo = t.id_tempo

    INNER JOIN mvp.gold.dim_orgao o
        ON f.id_orgao = o.id_orgao
),
valores AS (
    SELECT
        ano,

        MAX(CASE
            WHEN nome_orgao = 'CAPES'
            THEN dotacao_inicial
        END) AS capes_dotacao,

        MAX(CASE
            WHEN nome_orgao = 'MEC'
            THEN dotacao_inicial
        END) AS mec_dotacao

    FROM orcamento

    GROUP BY ano
)
SELECT
    ano,

    ROUND(
        (capes_dotacao / mec_dotacao) * 100,
        2
    ) AS participacao_capes_pct

FROM valores

ORDER BY ano;

ano,participacao_capes_pct
2013,5.22
2014,5.37
2015,6.06
2016,5.30
2017,4.61
2018,3.70
2019,3.46
2020,2.76


In [0]:
%sql

-- Compara a participação média da CAPES no orçamento do MEC entre 2013-2016 e 2017-2020.
WITH participacao AS (
    SELECT
        t.periodo,
        t.ano,

        MAX(CASE
            WHEN o.nome_orgao = 'CAPES'
            THEN f.dotacao_inicial
        END) AS capes_dotacao,

        MAX(CASE
            WHEN o.nome_orgao = 'MEC'
            THEN f.dotacao_inicial
        END) AS mec_dotacao

    FROM mvp.gold.fato_orcamento f

    INNER JOIN mvp.gold.dim_tempo t
        ON f.id_tempo = t.id_tempo

    INNER JOIN mvp.gold.dim_orgao o
        ON f.id_orgao = o.id_orgao

    GROUP BY
        t.periodo,
        t.ano
)

SELECT
    periodo,

    ROUND(
        AVG((capes_dotacao / mec_dotacao) * 100),
        2
    ) AS media_participacao_capes_pct

FROM participacao

GROUP BY periodo

ORDER BY periodo;

periodo,media_participacao_capes_pct
2013-2016,5.49
2017-2020,3.63


## Questão 4 — As alterações observadas no orçamento da CAPES foram acompanhadas por mudanças na quantidade e na distribuição regional das bolsas concedidas pela instituição?

In [0]:
%sql

-- Analisa a evolução anual da quantidade de bolsas concedidas.
SELECT
    t.ano,
    SUM(f.quantidade_bolsas) AS total_bolsas,

    ROUND(
        (
            SUM(f.quantidade_bolsas) /
            LAG(SUM(f.quantidade_bolsas))
                OVER (ORDER BY t.ano) - 1
        ) * 100,
        2
    ) AS variacao_anual_pct

FROM mvp.gold.fato_bolsas f

INNER JOIN mvp.gold.dim_tempo t
    ON f.id_tempo = t.id_tempo

GROUP BY t.ano
ORDER BY t.ano;

ano,total_bolsas,variacao_anual_pct
2013,85092,null
2014,105791,24.33
2015,105450,-0.32
2016,100433,-4.76
2017,101373,0.94
2018,101228,-0.14
2019,95290,-5.87
2020,95116,-0.18


In [0]:
%sql

-- Compara a participação regional das bolsas entre 2013-2016 e 2017-2020.
WITH bolsas_regiao AS (
    SELECT
        t.ano,
        t.periodo,
        r.nome_regiao AS regiao,
        f.quantidade_bolsas,

        SUM(f.quantidade_bolsas)
            OVER (PARTITION BY t.ano) AS total_ano

    FROM mvp.gold.fato_bolsas f

    INNER JOIN mvp.gold.dim_tempo t
        ON f.id_tempo = t.id_tempo

    INNER JOIN mvp.gold.dim_regiao r
        ON f.id_regiao = r.id_regiao
),
participacao AS (
    SELECT
        periodo,
        ano,
        regiao,
        (quantidade_bolsas / total_ano) * 100
            AS participacao_pct

    FROM bolsas_regiao
)
SELECT
    regiao,

    ROUND(
        AVG(CASE
            WHEN periodo = '2013-2016'
            THEN participacao_pct
        END),
        2
    ) AS media_2013_2016_pct,

    ROUND(
        AVG(CASE
            WHEN periodo = '2017-2020'
            THEN participacao_pct
        END),
        2
    ) AS media_2017_2020_pct,

    ROUND(
        AVG(CASE
            WHEN periodo = '2017-2020'
            THEN participacao_pct
        END)
        -
        AVG(CASE
            WHEN periodo = '2013-2016'
            THEN participacao_pct
        END),
        2
    ) AS variacao_pontos_percentuais

FROM participacao
GROUP BY regiao
ORDER BY media_2017_2020_pct DESC;

regiao,media_2013_2016_pct,media_2017_2020_pct,variacao_pontos_percentuais
SUDESTE,46.98,47.5,0.52
SUL,22.95,23.12,0.17
NORDESTE,18.25,17.73,-0.52
CENTRO-OESTE,7.55,7.08,-0.47
NORTE,4.27,4.58,0.31
